# ДЗ-9: ядро кредитного скоринга — обёртка и калькулятор суммы

1. Проверяем, что `app.core.model.AdvancedModel` даёт **такую же** вероятность дефолта, как прямой вызов `model.predict_proba()` на валидационной выборке.
2. Считаем экономику (ожидаемые издержки/прибыль/доход) для нескольких вариантов калькулятора суммы и сравниваем их, чтобы обосновать логику из `app.core.calculator.Calculator`.

## Настройка окружения

In [1]:
import json
import pickle
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# добавляем src/ в sys.path, чтобы работал `import app.core...`
sys.path.insert(0, str(Path("..", "src").resolve()))

from app.core.api import Features, build_feature_mappings
from app.core.model import AdvancedModel, SimpleModel

MODEL_PATH = Path("../models/catboost.pkl")
SCHEMA_PATH = Path("../docs/model_core/feature_schema.json")
VALIDATION_PATH = Path("../docs/model_core/validation_core_features.csv")

pd.set_option("display.float_format", "{:,.0f}".format)

## Загрузка данных и моделей

In [2]:
schema = json.loads(SCHEMA_PATH.read_text(encoding="utf-8"))
feature_order = schema["feature_order"]
field_to_column, column_to_field = build_feature_mappings(schema)

validation_df = pd.read_csv(VALIDATION_PATH)
print(f"Строк: {len(validation_df)}; колонок признаков: {len(feature_order)}")
validation_df.head()

Строк: 61503; колонок признаков: 99


,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_EMP_PHONE,...,EXT_SOURCE_1_was_missing,EXT_SOURCE_3_was_missing,OBS_30_CNT_SOCIAL_CIRCLE_was_missing,AMT_REQ_CREDIT_BUREAU_DAY_was_missing,interest_rate_was_missing,days_since_last_closed_credit_was_missing,days_since_last_active_credit_application_was_missing,NAME_TYPE_SUITE_was_missing,OCCUPATION_TYPE_was_missing,TARGET
0,1,"157,500","770,292",0,-13506,-105,"-2,876",-4402,19,1,...,1,0,0,0,0,0,0,0,0,0
1,0,"90,000","364,896",0,-17723,"-4,930","-4,654",-1259,NaN,1,...,0,0,0,0,0,1,0,0,0,0
2,0,"148,500","284,400",0,-24200,"-1,191","-12,511",-2474,NaN,1,...,1,0,0,0,0,0,1,0,0,0
3,0,"188,100","976,712",0,-9650,-355,-969,-2289,NaN,1,...,0,0,0,0,1,0,0,0,0,0
4,0,"180,000","323,194",0,-17487,"-1,366","-10,063",-1040,NaN,1,...,1,0,0,0,0,0,0,0,0,0


In [3]:
with open(MODEL_PATH, "rb") as pickled_model:
    raw_model = pickle.load(pickled_model)

advanced_model = AdvancedModel(str(MODEL_PATH), str(SCHEMA_PATH))
simple_model = SimpleModel(str(MODEL_PATH), str(SCHEMA_PATH))

## 2. Сверка proba: ядро vs предсказание модели напрямую

Строим `Features` для каждой строки валидационной выборки, считаем proba через
`AdvancedModel._predict_proba`, и сравниваем с прямым вызовом
`raw_model.predict_proba(validation_df[feature_order])[:, 1]`. Результаты должны
совпасть.

In [4]:
def row_to_features(row: pd.Series, column_to_field: dict[str, str]) -> Features:
    """Строит Features из строки датафрейма валидационной выборки.

    Args:
        row: строка validation_df (одна заявка), включая TARGET —
            лишние ключи (TARGET, SK_ID_CURR, если есть) игнорируются
            Features.from_record по списку полей дата-класса.
        column_to_field: сопоставление имя_колонки -> имя_поля
            (см. app.core.api.build_feature_mappings).

    Returns:
        Features: признаки одной заявки.
    """
    return Features.from_record(row.to_dict(), column_to_field)


core_proba = validation_df.apply(
    lambda row: advanced_model._predict_proba(row_to_features(row, column_to_field)),
    axis=1,
).to_numpy()

direct_proba = raw_model.predict_proba(validation_df[feature_order])[:, 1]

max_abs_diff = np.max(np.abs(core_proba - direct_proba))
print(f"Максимальное расхождение proba: {max_abs_diff:.2e}")
assert np.allclose(core_proba, direct_proba), "Proba ядра и прямого вызова модели разошлись!"

Максимальное расхождение proba: 0.00e+00


## 3. Сравнение схем одобрения суммы

Строим таблицу в формате примера из лекции:
Сумма / вероятность дефолта / Выдачи / Дефолт / Вернут / Издержки / прибыль / Доход / Итог —
но вместо ожидаемого числа дефолтов используем **фактический** `TARGET`
валидационной выборки.

Сравниваем три схемы:
1. `baseline_full` — одобрять 100% запрошенной суммы всем, кто прошёл порог 0.3.
2. `proposed_calculator` — `app.core.calculator.Calculator` (ступени 0.5x/1.0x/1.5x от p*).
3. `lecture_fixed_tiers` — схема из лекции (жёсткие суммы 100k/200k/400k/500k) для сравнения.

In [5]:
from app.core.calculator import Calculator

MARGIN_RATE = schema["fallback_interest_rate"]
LGD = 1.0


def economics_report(
    proba: np.ndarray,
    target: np.ndarray,
    approved_amount: np.ndarray,
    rate: np.ndarray,
    n_buckets: int = 10,
) -> pd.DataFrame:
    """Считает таблицу экономики по децилям PD (аналог Пример экономики.xlsx).

    Args:
        proba: вероятность дефолта по каждой заявке.
        target: фактический TARGET (1 — дефолт, 0 — возврат).
        approved_amount: одобренная сумма по каждой заявке (0 — отказ).
        rate: эффективная ставка маржи по каждой заявке (interest_rate
            клиента или fallback).
        n_buckets: число децилей PD для агрегации отчёта.

    Returns:
        pd.DataFrame: таблица с колонками Выдачи/Дефолт/Вернут/Издержки/
            прибыль/Доход по децилям PD + итоговая строка "Итог".
    """
    frame = pd.DataFrame(
        {
            "proba": proba,
            "target": target,
            "amount": approved_amount,
            "rate": rate,
        }
    )
    frame = frame[frame["amount"] > 0].copy()
    frame["pd_decile"] = pd.qcut(frame["proba"], n_buckets, duplicates="drop")

    frame["Издержки"] = np.where(frame["target"] == 1, -frame["amount"] * LGD, 0.0)
    frame["прибыль"] = np.where(frame["target"] == 0, frame["amount"] * frame["rate"], 0.0)
    frame["Доход"] = frame["Издержки"] + frame["прибыль"]

    report = frame.groupby("pd_decile", observed=True).agg(
        Выдачи=("amount", "count"),
        Сумма_среднее=("amount", "mean"),
        Дефолт=("target", "sum"),
        Издержки=("Издержки", "sum"),
        прибыль=("прибыль", "sum"),
        Доход=("Доход", "sum"),
    )
    report.loc["Итог"] = report.sum(numeric_only=True)
    return report


effective_rate = validation_df["interest_rate"].fillna(MARGIN_RATE).to_numpy()
target = validation_df["TARGET"].to_numpy()
requested_amount = validation_df["AMT_CREDIT"].to_numpy()
accepted_mask = core_proba < SimpleModel._threshold

#### Схема 1: baseline — одобрять 100% запрошенной суммы всем принятым

In [6]:
baseline_amount = np.where(accepted_mask, requested_amount, 0.0)

report_baseline = economics_report(core_proba, target, baseline_amount, effective_rate)
report_baseline

,Выдачи,Сумма_среднее,Дефолт,Издержки,прибыль,Доход
pd_decile,,,,,,
"(0.0172, 0.117]","2,181","621,536",21,"-7,531,254","516,882,249","509,350,995"
"(0.117, 0.144]","2,180","636,246",18,"-10,562,612","531,199,575","520,636,963"
"(0.144, 0.167]","2,180","652,375",29,"-16,812,058","552,496,641","535,684,583"
"(0.167, 0.188]","2,180","650,077",38,"-27,106,862","562,186,441","535,079,579"
"(0.188, 0.207]","2,180","646,474",60,"-39,240,446","569,322,220","530,081,774"
"(0.207, 0.225]","2,180","644,155",53,"-33,528,110","567,164,229","533,636,119"
"(0.225, 0.244]","2,180","645,727",59,"-37,523,228","593,806,691","556,283,463"
"(0.244, 0.262]","2,180","629,346",64,"-37,548,468","578,254,893","540,706,425"
"(0.262, 0.281]","2,180","632,087",61,"-36,154,170","595,666,945","559,512,775"


#### Схема 2: Calculator (ступени x/1.2x/2x от p*)

In [7]:
calculator = Calculator(fallback_interest_rate=MARGIN_RATE)

proposed_amount = np.array(
    [
        calculator.calc_amount(proba=p, features=row_to_features(row, column_to_field))
        for p, (_, row) in zip(core_proba, validation_df.iterrows(), strict=True)
    ]
)

report_proposed = economics_report(core_proba, target, proposed_amount, effective_rate)
report_proposed

,Выдачи,Сумма_среднее,Дефолт,Издержки,прибыль,Доход
pd_decile,,,,,,
"(0.0172, 0.152]","5,024","616,882",49,"-20,826,790","1,205,876,938","1,185,050,148"
"(0.152, 0.2]","5,024","624,150",99,"-65,661,689","1,289,216,935","1,223,555,246"
"(0.2, 0.244]","5,023","578,003",128,"-72,538,374","1,289,626,100","1,217,087,726"
"(0.244, 0.288]","5,024","512,221",152,"-76,966,142","1,224,197,152","1,147,231,010"
"(0.288, 0.335]","5,024","418,945",188,"-68,639,515","1,053,452,393","984,812,878"
"(0.335, 0.388]","5,023","308,744",229,"-60,205,185","833,853,360","773,648,175"
"(0.388, 0.446]","5,024","221,038",308,"-66,407,730","615,892,782","549,485,052"
"(0.446, 0.517]","5,023","142,969",393,"-58,590,791","373,804,845","315,214,054"
"(0.517, 0.611]","5,024","117,906",504,"-59,588,888","311,111,388","251,522,500"


#### Схема 3: фиксированные суммы (для сравнения) — 100k/200k/400k/500k

In [ ]:
# Схема 3: фиксированные суммы из лекции (для сравнения) — 100k/200k/400k/500k
def lecture_fixed_amount(proba: float) -> float:
    """Повторяет калькулятор из лекции с фиксированными суммами.

    Args:
        proba: вероятность дефолта заявки.

    Returns:
        float: фиксированная сумма одобрения (без учёта признаков клиента).
    """
    if proba < 0.10:
        return 400_000.0
    if proba < 0.20:
        return 200_000.0
    return 100_000.0


lecture_amount = np.where(accepted_mask, [lecture_fixed_amount(p) for p in core_proba], 0.0)

report_lecture = economics_report(core_proba, target, lecture_amount, effective_rate)
report_lecture

,Выдачи,Сумма_среднее,Дефолт,Издержки,прибыль,Доход
pd_decile,,,,,,
"(0.0172, 0.117]","2,181","313,159",21,"-7,000,000","257,297,853","250,297,853"
"(0.117, 0.144]","2,180","200,000",18,"-3,600,000","171,061,072","167,461,072"
"(0.144, 0.167]","2,180","200,000",29,"-5,800,000","173,267,023","167,467,023"
"(0.167, 0.188]","2,180","200,000",38,"-7,600,000","176,271,668","168,671,668"
"(0.188, 0.207]","2,180","166,330",60,"-10,300,000","148,268,340","137,968,340"
"(0.207, 0.225]","2,180","100,000",53,"-5,300,000","89,067,085","83,767,085"
"(0.225, 0.244]","2,180","100,000",59,"-5,900,000","93,008,564","87,108,564"
"(0.244, 0.262]","2,180","100,000",64,"-6,400,000","92,437,580","86,037,580"
"(0.262, 0.281]","2,180","100,000",61,"-6,100,000","94,389,308","88,289,308"


In [9]:
comparison = pd.DataFrame(
    {
        "baseline_full (100%)": report_baseline.loc["Итог"],
        "proposed_calculator": report_proposed.loc["Итог"],
        "lecture_fixed_tiers": report_lecture.loc["Итог"],
    }
).T
comparison

,Выдачи,Сумма_среднее,Дефолт,Издержки,прибыль,Доход
baseline_full (100%),"21,802","6,384,251",484,"-300,314,236","5,647,395,494","5,347,081,258"
proposed_calculator,"50,237","3,645,104","2,964","-643,754,703","8,485,135,072","7,841,380,369"
lecture_fixed_tiers,"21,802","1,579,489",484,"-66,100,000","1,389,597,471","1,323,497,471"


## Выводы

- **Proba ядра совпадает с прямым вызовом `predict_proba()`**. Обёртка корректно воспроизводит модель.
- **Пороги `Calculator` (1.0x/1.2x/2.0x от `p*`) подбирались эмпирически, в несколько итераций**: начала с `0.5x/1.0x/1.5x`, финально — `1.0x/1.2x/2.0x`.
- **`baseline_full` vs `lecture_fixed_tiers`** на одной и той же популяции, 21,802 заявки, `baseline_full` даёт Доход 5,347 млн против 1,323 млн у фиксированных сумм, т.к. эта схема занижает суммы заведомо безопасным клиентам, не учитывая их признаки.
- **`proposed_calculator` в этом прогоне посчитан без гейта `0.3`** (Calculator сам отсеивает и выдаёт суммы для всей выборки, 50,237 из ~61,500 заявок, а не только для 21,802 прошедших гейт) — поэтому его `Итог` (7,841 млн) напрямую не сравним с `baseline_full` по той же базе. Тем не менее калькулятор показывает, стоит ли вообще заменить строгий порог по модели на калькулятор.
- Несмотря на разные популяции, результат показателен: калькулятор без гейта реализует на порядок больше дефолтов (2,964 против 484) и издержек (-643.8 млн против -300.3 млн), но итоговый Доход всё равно почти в 1.5 раза выше baseline — т.к. гейт `0.3` отсекает много клиентов, которые по персональной экономике всё ещё прибыльны.
- **Финальное решение**: калькулятор оценивает каждого клиента по отдельности, как будто дефолты независимы друг от друга, тогда как в реальности они могут коррелировать. Широкий риск-диапазон может быть в среднем прибыльным на спокойных исторических данных и обанкротить портфель в плохой год. Так же, если совсем убрать отсечку, которая есть в baseline, то любые ошибки в расчёте фичи могут повлечь реальные убытки. С помощью порога мы страхуемся от таких случаев. Однако, наши расчёты показали, что можно пересмотреть его значение 0.3 в сторону увеличения.
